<a href="https://colab.research.google.com/github/andrewgodbout/MCS-3950-F26/blob/main/Notebooks/AL07-CNNTrain/AL7_CNN_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AL7 — Training a CNN: Learning Rate, Depth and Augmentation
**MCS 3950 Computer Vision · UPEI · Fall 2026 · Friday Nov 6**

In L12 you trained a 1,898-weight CNN on 8 × 8 digits. In L13 you met the famous networks
and the two ideas that make deep networks trainable: **batch normalisation** and **skip
connections**. Today you train your own CNN on real photographs (tiny ones) and turn the
three knobs that matter most in practice:

1. the **learning rate**: too small, just right, too big;
2. **depth**: does adding layers help, and what does batch norm change?
3. **data augmentation**: the cheapest fix for over-fitting.

The data is **CIFAR-10**: 32 × 32 colour photos in 10 classes. We use a 14,000-image slice
of it so that every training run takes seconds on a GPU.

| Part | Topic | Time |
|------|-------|------|
| 0 | Setup, and *you* vs. the network | ~5 min |
| 1 | The training loop and the learning rate | ~13 min |
| 2 | Deeper networks and batch norm | ~11 min |
| 3 | Over-fitting and data augmentation | ~13 min |
| 4 | The test set, once | ~5 min |
| ★ | Bonus: forgetting `model.eval()` | if time |

There are **three tasks** and a few **checkpoint questions**. Before each experiment there
is a 🔮 **Predict first** prompt: write your guess down *before* you run the cell.

> **Use a GPU.** In Colab: **Runtime → Change runtime type → T4 GPU**, then run the setup
> cell. With a GPU all the training today takes a few minutes; on a CPU it is closer to
> 20, so if you can't get a GPU, pair up with someone who has one.
>
> **Not submitted or graded.** This is Test 3 material (L12–L15), and it is the warm-up
> for **Assignment 2** (CNN implementation and training, assigned Mon Nov 9).

In [ ]:
import io, os, time, copy, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})
GOLD, RED, TEAL, GREEN, NAVY, GREY = "#C8962C", "#B82828", "#1A7A8A", "#2E7D50", "#1A3A5C", "#9AA5B1"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} on {DEVICE.upper()}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))
if DEVICE == "cpu":
    print("No GPU. Colab: Runtime -> Change runtime type -> T4 GPU, then re-run this cell.\n"
          "On a CPU, the training in this lab takes about 20 minutes in total.")

## Part 0 — The data, and you vs. the network  `⏱ ~5 min`

`AL7_cifar10.npz` (~41 MB) holds three balanced slices of CIFAR-10:

| Split | Images | Per class | Used for |
|---|---|---|---|
| `train` | 10,000 | 1,000 | learning the weights |
| `val` | 2,000 | 200 | choosing settings (learning rate, depth, augmentation) |
| `test` | 2,000 | 200 | **once**, at the very end (Part 4) |

The full dataset has 50,000 training images. We use a fifth of them so that each run is
quick, and so that over-fitting shows up early enough to study it in Part 3.

In [ ]:
BASE = ("https://raw.githubusercontent.com/andrewgodbout/"
        "MCS-3950-F26/main/Notebooks/data/")
FILE = "AL7_cifar10.npz"
if not os.path.exists(FILE):
    with urllib.request.urlopen(BASE + FILE, timeout=180) as r:
        blob = r.read()
    with open(FILE, "wb") as f:
        f.write(blob)
    print(f"{FILE}: {len(blob)/1e6:.1f} MB downloaded")
else:
    print(f"{FILE}: already present")

D = np.load(FILE)
CLASSES = [str(c) for c in D["classes"]]
print({k: D[k].shape for k in ("x_train", "x_val", "x_test")})
print("classes:", ", ".join(f"{i} {c}" for i, c in enumerate(CLASSES)))

### 🔮 You vs. the network

Below are 16 validation images at their real size, 32 × 32 pixels, blown up. **Write down
your guess for each one** (number → class) before you run the cell after it. In 2011, Andrej
Karpathy labelled 400 CIFAR-10 images by hand and scored about **94%**. The best computer
then managed about 80%. Today a well-tuned network passes 94% after a few seconds of
training on the full dataset. Ours will be smaller and see a fifth of the data. In Part 4 it
answers the same 16.

In [ ]:
# 16 validation images: one of every class, plus a second bird, cat, deer, dog, horse and truck.
yv = D["y_val"]
HOOK = np.random.default_rng(6).permutation([np.where(yv == c)[0][0] for c in range(10)]
                                           + [np.where(yv == c)[0][1] for c in (2, 3, 4, 5, 7, 9)])
fig, axes = plt.subplots(2, 8, figsize=(16, 4.6), gridspec_kw={"hspace": 0.25})
for n, (k, ax) in enumerate(zip(HOOK, axes.ravel())):
    ax.imshow(D["x_val"][k], interpolation="nearest"); ax.axis("off")
    ax.set_title(f"#{n + 1}", fontsize=12)
plt.suptitle("Guess each one: " + ", ".join(CLASSES), y=0.98, fontsize=12)
plt.show()

In [ ]:
# Run this only after you have written down your 16 guesses.
print("  ".join(f"#{n + 1} {CLASSES[D['y_val'][k]]}" for n, k in enumerate(HOOK)))

### To tensors

PyTorch wants images as `(N, channels, height, width)` floats. We scale to 0–1, then subtract
each colour channel's mean and divide by its standard deviation (computed on `train` only).
That is L9's **centring**, once more: every input channel then has mean 0 and spread 1, which
keeps the first layer's outputs in a sensible range. Everything goes onto the GPU once, so
training never waits for data.

In [ ]:
def to_tensor(x):
    return torch.tensor(x).permute(0, 3, 1, 2).float().div(255)   # (N, 32, 32, 3) uint8 -> (N, 3, 32, 32)

Xtr, Xva, Xte = to_tensor(D["x_train"]), to_tensor(D["x_val"]), to_tensor(D["x_test"])
MEAN = Xtr.mean(dim=(0, 2, 3), keepdim=True)
STD = Xtr.std(dim=(0, 2, 3), keepdim=True)
Xtr, Xva, Xte = [((x - MEAN) / STD).to(DEVICE) for x in (Xtr, Xva, Xte)]
ytr, yva, yte = [torch.tensor(D[k]).long().to(DEVICE) for k in ("y_train", "y_val", "y_test")]

def to_image(x):
    # One normalised (3, 32, 32) tensor -> (32, 32, 3) array in 0..1 for plt.imshow.
    return (x.cpu() * STD[0] + MEAN[0]).clamp(0, 1).permute(1, 2, 0).numpy()

print("channel means (R, G, B):", MEAN.ravel().numpy().round(3), " stds:", STD.ravel().numpy().round(3))
print("Xtr:", tuple(Xtr.shape), Xtr.dtype, "on", Xtr.device)

---
## Part 1 — The training loop and the learning rate  `⏱ ~13 min`

### Step 1 · The network

The L12 digit CNN, one size up: three rounds of **conv 3 × 3 → ReLU → max-pool**, so the
image shrinks 32 → 16 → 8 → 4 while the number of channels grows 32 → 64 → 128. Then,
as in ResNet (L13), **global average pooling** turns each of the 128 final 4 × 4 maps into
one number, and a fully connected layer turns those 128 numbers into 10 class scores.

In [ ]:
def small_cnn():
    return nn.Sequential(
        nn.Conv2d(3, 32, 3, padding=1),   nn.ReLU(), nn.MaxPool2d(2),   # (3, 32, 32)  -> (32, 16, 16)
        nn.Conv2d(32, 64, 3, padding=1),  nn.ReLU(), nn.MaxPool2d(2),   #              -> (64, 8, 8)
        nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   #              -> (128, 4, 4)
        nn.AdaptiveAvgPool2d(1), nn.Flatten(),                          # global average pooling -> 128
        nn.Linear(128, 10))                                             # 10 class scores

def n_params(model):
    return sum(p.numel() for p in model.parameters())

torch.manual_seed(0)
net = small_cnn()
x = torch.zeros(1, 3, 32, 32)
print(f"{'layer':<20}{'output shape':>18}{'parameters':>12}")
for layer in net:
    x = layer(x)
    print(f"{layer.__class__.__name__:<20}{str(tuple(x.shape[1:])):>18}{n_params(layer):>12,}")
print(f"{'total':<20}{'':>18}{n_params(net):>12,}")

**By hand (L12 slide 21):** a conv layer with $k \times k$ filters, $C_\text{in}$ input
channels and $C_\text{out}$ filters has $C_\text{out}(k^2 C_\text{in} + 1)$ parameters.
Check the first one: $32 \times (9 \times 3 + 1) = 896$. Now do the second conv layer in your
head, then check the table.

### Step 2 · One training step  (Task 1)

This is the loop from L12 slide 18. One **step** takes one mini-batch of 128 images:

1. **forward**: run the batch through the network to get a `(128, 10)` array of scores.
   Calling the model on the images does this: `scores = model(xb)`.
2. **loss**: cross-entropy between the scores and the true labels (L12 §2). For
   classification we want the *probability* the network gives each class, turned into an
   error measure. `F.cross_entropy` does both: a softmax turns the raw scores into
   probabilities, then $-\ln P(\text{true class})$ turns that into a loss. So pass it the
   **raw** scores, not probabilities.
3. **clear** the old gradients (PyTorch adds new gradients to old ones unless you do);
4. **backward**: backpropagation fills in $\partial L / \partial w$ for every weight;
5. **update**: the optimiser moves every weight a little downhill.

We use **SGD with momentum 0.9**. It is L12's update $w \leftarrow w - \eta\,\partial L/\partial w$,
except that the direction is a running average of recent gradients rather than the latest
one alone: a ball rolling downhill rather than a walker who re-plans every step. The ball
gains speed when the slope keeps pointing the same way. The learning rate $\eta$ still sets
the step size.

**Task 1:** complete `train_step`. It returns two plain numbers:
* the batch's loss as a Python float, `loss.item()`;
* the number of images it classified correctly: how many rows of `scores` have their
  largest score at the true label in `yb`.

(`.item()` pulls the single number out of a one-element tensor.)

In [ ]:
def train_step(model, opt, xb, yb):
    '''
    One step of mini-batch gradient descent.
      model : the network          opt : a torch.optim optimiser for model's parameters
      xb    : (B, 3, 32, 32) images  yb : (B,) labels 0..9
    Returns (loss as a float, number of correct predictions in the batch).
    '''
    # ── YOUR CODE HERE ──────────────────────────────────────────────────────────
    # 1. scores = model(xb)                      (B, 10)
    # 2. loss = F.cross_entropy(scores, yb)
    # 3. opt.zero_grad()
    # 4. loss.backward()
    # 5. opt.step()
    # return loss.item(), (scores.argmax(dim=1) == yb).sum().item()
    # ────────────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Task 1: write the five lines of a training step.")

The rest of the machinery is given. `fit` runs `train_step` over the whole training set
once per **epoch**, in a fresh random order each time, and after every epoch measures
accuracy on `val`. Note the two mode switches: `model.train()` before training and
`model.eval()` before measuring. They matter as soon as there is batch norm in the
network (Part 2, and the bonus).

In [ ]:
@torch.no_grad()
def predict(model, X, bs=500):
    model.eval()
    return torch.cat([model(X[i:i + bs]).argmax(dim=1) for i in range(0, len(X), bs)])

def accuracy(model, X, Y):
    return (predict(model, X) == Y).float().mean().item()

def fit(model, lr, epochs, augment=None, decay=False, bs=128, seed=0, label=""):
    '''Train with SGD (momentum 0.9). Returns a dict of per-epoch histories.
    decay=True lowers the learning rate in a straight line from lr to 0 over the run (Part 3).'''
    torch.manual_seed(seed)
    model.to(DEVICE)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    n_steps = epochs * ((len(Xtr) + bs - 1) // bs)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda k: 1 - k / n_steps) if decay else None
    hist = {"loss": [], "train_acc": [], "val_acc": []}
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr), device=DEVICE)
        total, right = 0.0, 0
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i + bs]
            xb, yb = Xtr[idx], ytr[idx]
            if augment is not None:
                xb = augment(xb)
            loss, n_right = train_step(model, opt, xb, yb)
            if sched is not None:
                sched.step()
            total += loss * len(idx); right += n_right
        hist["loss"].append(total / len(Xtr))
        hist["train_acc"].append(right / len(Xtr))
        hist["val_acc"].append(accuracy(model, Xva, yva))
        if not np.isfinite(hist["loss"][-1]):          # the weights have blown up: stop early
            print(f"  {label}: loss is {hist['loss'][-1]} after epoch {ep + 1}. Diverged; stopping.")
            break
    hist["time"] = time.time() - t0
    hist["model"] = model
    print(f"{label:<22} final training loss {hist['loss'][-1]:6.3f}   train acc {hist['train_acc'][-1]:5.1%}"
          f"   val acc {hist['val_acc'][-1]:5.1%}   ({hist['time']:.0f} s)")
    return hist

def plot_runs(runs, labels, colours, title, train_acc=False):
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
    for lab, col in zip(labels, colours):
        h = runs[lab]; ep = np.arange(1, len(h["loss"]) + 1)
        ax[0].plot(ep, h["loss"], "o-", color=col, lw=2.2, ms=4, label=lab)
        ax[1].plot(ep, h["val_acc"], "o-", color=col, lw=2.2, ms=4, label=lab + " (val)")
        if train_acc:
            ax[1].plot(ep, h["train_acc"], "--", color=col, lw=1.6, label=lab + " (train)")
    ax[0].axhline(np.log(10), color=GREY, ls=":"); ax[0].text(1, np.log(10) + 0.07, "ln 10 = 2.30: guessing", color=GREY)
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("training loss"); ax[0].set_ylim(0, 2.6)
    ax[1].axhline(0.1, color=GREY, ls=":"); ax[1].text(1, 0.115, "guessing (10%)", color=GREY)
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("accuracy"); ax[1].set_ylim(0, 1)
    ax[0].set_title("Training loss", fontsize=12); ax[1].set_title("Accuracy", fontsize=12)
    ax[1].legend(fontsize=9, loc="upper left", ncol=2 if train_acc else 1)
    plt.suptitle(title, fontsize=13); plt.tight_layout(); plt.show()

RUNS = {}

### Step 3 · Sanity check: the loss before any training

🔮 **Predict first.** An untrained network knows nothing, so it should give each of the 10
classes probability about $\frac{1}{10}$. What cross-entropy loss is that? (L12 slide 12:
loss $= -\ln P(\text{true class})$.)

In [ ]:
torch.manual_seed(0)
net = small_cnn().to(DEVICE)
with torch.no_grad():
    scores = net(Xtr[:500])
    P = scores.softmax(dim=1)
print(f"loss before training: {F.cross_entropy(scores, ytr[:500]).item():.3f}    "
      f"(-ln(1/10) = {np.log(10):.3f})")
print(f"probabilities the untrained network gives image 0: {P[0].cpu().numpy().round(3)}")
try:
    opt = torch.optim.SGD(net.parameters(), lr=0.01, momentum=0.9)
    print("one train_step:", train_step(net, opt, Xtr[:128], ytr[:128]), " <- (loss, correct out of 128)")
except NotImplementedError:
    print("Complete Task 1 first.")

This is the first thing to check whenever you train a classifier. A starting loss far from
$\ln C$ (for $C$ classes) means something is wrong before training has even begun: labels
off by one, scores already squashed by a softmax, unnormalised inputs. In L13, the plain
31-layer network sat at exactly 2.303 for 30 epochs: it never got past this point.

### Step 4 · The learning rate

L12 slide 14 had three step sizes on a one-weight problem: too small (crawls), about right,
and too big (overshoots, then climbs out of the valley). Here are four learning rates on a
94,538-weight network, 8 epochs each.

🔮 **Predict first.** Which learning rate will have the best validation accuracy after 8
epochs? Will any of them fail completely?

In [ ]:
LRS = [0.002, 0.01, 0.05, 0.3]
try:
    for lr in LRS:
        torch.manual_seed(0)
        RUNS[f"lr {lr}"] = fit(small_cnn(), lr=lr, epochs=8, label=f"lr {lr}")
except NotImplementedError:
    print("Complete Task 1 first.")

In [ ]:
if "lr 0.3" in RUNS:
    plot_runs(RUNS, [f"lr {lr}" for lr in LRS], [GREY, TEAL, NAVY, RED],
              "Same network, same data, four learning rates (SGD, momentum 0.9, 8 epochs)")

In [ ]:
# What is left inside each network? The share of ReLU outputs that are exactly 0, layer by layer,
# on 500 validation images. (A healthy ReLU layer is off for some inputs and on for others.)
def zero_share(model, X):
    model.eval(); x = X; out = []
    with torch.no_grad():
        for layer in model:
            x = layer(x)
            if isinstance(layer, nn.ReLU):
                out.append((x == 0).float().mean().item())
    return out

for lab in ("lr 0.05", "lr 0.3"):
    if lab not in RUNS:
        continue
    if not np.isfinite(RUNS[lab]["loss"][-1]):
        print(f"{lab:8}: weights are nan, nothing left to measure")
    else:
        print(f"{lab:8}: ReLU outputs that are 0 in layers 1, 2, 3:",
              "  ".join(f"{z:6.1%}" for z in zero_share(RUNS[lab]["model"], Xva[:500])))

### Checkpoint 1

**Q1a.** Match each learning rate to one of L12's three pictures (crawls / about right /
overshoots). For the slowest one, what would you have to do to reach the same accuracy as the
best one, and why is that a bad deal?

*Your answer:*

**Q1b.** $\eta = 0.3$ failed. Depending on the machine, its loss either becomes `nan`
(not a number) or drops back to 2.30 (guessing) and stays there; the cell above shows which
happened here. Using $w \leftarrow w - \eta\,\partial L/\partial w$, explain how a few
steps that are too big can throw the weights far out of the valley. Then explain why neither
failure can recover: (a) once a weight is `nan`; (b) once a ReLU layer outputs 0 for every image
(L12's "dead ReLU"): what is the gradient of every weight before that layer?

*Your answer:*

---
## Part 2 — Deeper networks and batch norm  `⏱ ~11 min`

VGG's lesson (L13 §2): stack more 3 × 3 convolutions. ResNet's lesson (L13 §3): a deeper
*plain* network can be **harder to train**, and batch norm and skip connections fix that.
Let's see how much of that shows up already at six layers.

### Step 1 · A network builder  (Task 2)

**Task 2:** write `make_cnn(per_stage, bn)`. It builds the same three stages as `small_cnn`
(widths 32, 64, 128, each stage ending in a 2 × 2 max-pool), but each stage now has
`per_stage` conv layers instead of one. With `bn=True`, every conv layer is followed by a
`nn.BatchNorm2d` **before** its ReLU, as in L13 slide 18 and in ResNet:

    conv 3×3  →  [BatchNorm2d]  →  ReLU

So `make_cnn(1, False)` is exactly `small_cnn()`, and `make_cnn(2, True)` has six conv
layers, each with batch norm.

In [ ]:
def make_cnn(per_stage=1, bn=False, widths=(32, 64, 128)):
    '''
    Three stages; each stage = per_stage x [conv 3x3 (padding 1) -> (BatchNorm2d) -> ReLU], then MaxPool2d(2).
    Then global average pooling, Flatten, and Linear(last width, 10).
    '''
    layers, c_in = [], 3
    # ── YOUR CODE HERE ──────────────────────────────────────────────────────────
    # for w in widths:
    #     repeat per_stage times:
    #         layers.append(nn.Conv2d(c_in, w, 3, padding=1))
    #         if bn: layers.append(nn.BatchNorm2d(w))
    #         layers.append(nn.ReLU());  c_in = w
    #     layers.append(nn.MaxPool2d(2))
    # layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c_in, 10)]
    # ────────────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Task 2: build the stages.")
    return nn.Sequential(*layers)

In [ ]:
try:
    torch.manual_seed(0); a = small_cnn()
    torch.manual_seed(0); b = make_cnn(1, False)
    x = torch.randn(4, 3, 32, 32)
    with torch.no_grad():
        same = torch.allclose(a(x), b(x))
    print(f"make_cnn(1, False) == small_cnn(): {same}   ({n_params(b):,} parameters)")
    for ps, bn in ((2, False), (2, True)):
        m = make_cnn(ps, bn)
        n_conv = sum(isinstance(l, nn.Conv2d) for l in m)
        n_bn = sum(isinstance(l, nn.BatchNorm2d) for l in m)
        print(f"make_cnn({ps}, {bn!s:5}): {n_conv} conv layers, {n_bn} batch-norm layers, {n_params(m):,} parameters")
except NotImplementedError:
    print("Complete Task 2 first.")

**By hand:** each `BatchNorm2d(C)` learns one scale $\gamma$ and one shift $\beta$ per channel
(L13 slide 18), so $2C$ parameters. The six layers have $C$ = 32, 32, 64, 64, 128, 128. How many
parameters does batch norm add in total? Check against the two lines above.

### Step 2 · Three networks, same learning rate, same 8 epochs

We keep $\eta = 0.05$ from Part 1 and compare:

* **3 conv**: `small_cnn`, already trained in Part 1 (`lr 0.05`);
* **6 conv, plain**: twice as deep;
* **6 conv + BN**: twice as deep, with batch norm.

🔮 **Predict first.** Rank the three by validation accuracy after 8 epochs. Does doubling
the depth help on its own?

In [ ]:
try:
    RUNS["3 conv"] = RUNS["lr 0.05"]
    torch.manual_seed(0); RUNS["6 conv, plain"] = fit(make_cnn(2, False), lr=0.05, epochs=8, label="6 conv, plain")
    torch.manual_seed(0); RUNS["6 conv + BN"] = fit(make_cnn(2, True), lr=0.05, epochs=8, label="6 conv + BN")
except (NotImplementedError, KeyError):
    print("Complete Tasks 1 and 2 (and the Part 1 learning-rate runs) first.")

In [ ]:
if "6 conv + BN" in RUNS:
    plot_runs(RUNS, ["3 conv", "6 conv, plain", "6 conv + BN"], [NAVY, RED, GREEN],
              "Depth, with and without batch norm (lr 0.05, 8 epochs)")

### Checkpoint 2

**Q2a.** The plain 6-layer network is *worse* than the 3-layer one, and not only on
`val`: its **training** loss is higher too. Is that over-fitting? What is it instead, and which
L13 slide does it echo? (Look at its first epoch.)

*Your answer:*

**Q2b.** In one or two sentences: what does `BatchNorm2d` do to each channel of each
mini-batch, and why does that make the 6-layer network easier to train? (L13 slide 18 by hand:
7, 9, 10, 11, 13 → −1.5 … 1.5.)

*Your answer:*

---
## Part 3 — Over-fitting and data augmentation  `⏱ ~13 min`

### Step 1 · Train longer and watch the gap

The best network so far is **6 conv + BN**. Two changes for a longer run: `EPOCHS` epochs
instead of 8, and `decay=True`. With decay, `fit` lowers the learning rate in a straight line,
from 0.05 at the first step to 0 at the last. That is Part 1's picture again: big steps early
to travel across the valley, small steps at the end to settle into the bottom of it instead
of bouncing from side to side. We plot **training** accuracy (dashed) next to **validation**
accuracy (solid).

🔮 **Predict first.** At the end, roughly what training accuracy, and what validation
accuracy?

In [ ]:
EPOCHS = 25
try:
    torch.manual_seed(0)
    RUNS["no augmentation"] = fit(make_cnn(2, True), lr=0.05, epochs=EPOCHS, decay=True, label="no augmentation")
    plot_runs(RUNS, ["no augmentation"], [NAVY], f"6 conv + BN, {EPOCHS} epochs: training vs. validation accuracy",
              train_acc=True)
except NotImplementedError:
    print("Complete Tasks 1 and 2 first.")

The network is starting to **memorise** its 10,000 training photos: training accuracy keeps
climbing, validation accuracy levels off, and the gap between them is over-fitting. The
fix with the best return is **more data**. We can't get more photos, but we can show the
network a slightly different version of each photo every time it sees it.

### Step 2 · Data augmentation  (Task 3)

An augmentation must keep the label true. A cat flipped left-to-right is still a cat; a cat
shifted 3 pixels is still a cat. Every epoch, each training image gets:

* a **horizontal flip**, with probability ½;
* a **random shift** of up to 4 pixels in x and y: pad 4 pixels of zeros on every side
  (40 × 40), then cut out a random 32 × 32 window. (Zero is the *mean* colour after
  normalising, so the padding is a dull grey border.)

**Task 3:** complete `augment`. It gets a batch on the GPU and returns a new batch of the
same shape. Only the training batches are augmented; `val` and `test` are never touched.

In [ ]:
def augment(x, max_shift=4):
    '''
    x : (B, 3, 32, 32) batch.  Returns a new (B, 3, 32, 32) batch where each image is
    flipped left-right with probability 1/2, then shifted by up to max_shift pixels.
    '''
    B = len(x)
    # ── YOUR CODE HERE ──────────────────────────────────────────────────────────
    # 1. flip = torch.rand(B, device=x.device) < 0.5                 (B,) True/False
    #    x = torch.where(flip[:, None, None, None], x.flip(dims=[3]), x)     dim 3 is width
    # 2. p = F.pad(x, (max_shift, max_shift, max_shift, max_shift))   (B, 3, 40, 40)
    #    dx = torch.randint(0, 2*max_shift + 1, (B,)).tolist(); same for dy
    #    out = torch.empty_like(x)
    #    for i in range(B): out[i] = p[i, :, dy[i]:dy[i] + 32, dx[i]:dx[i] + 32]
    # ────────────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Task 3: flip and shift.")

In [ ]:
# Look before you train: one training image, and seven augmented copies of it.
try:
    torch.manual_seed(1)
    k = 7
    copies = augment(Xtr[k:k + 1].repeat(7, 1, 1, 1))
    fig, axes = plt.subplots(1, 8, figsize=(16, 2.6))
    axes[0].imshow(to_image(Xtr[k]), interpolation="nearest"); axes[0].set_title(f"original ({CLASSES[ytr[k]]})")
    for ax, c in zip(axes[1:], copies):
        ax.imshow(to_image(c), interpolation="nearest"); ax.set_title("augmented")
    for ax in axes: ax.axis("off")
    plt.tight_layout(); plt.show()
except NotImplementedError:
    print("Complete Task 3 first.")

### Step 3 · Same network, same number of epochs, augmented

🔮 **Predict first.** With augmentation, will **training** accuracy go up or down? And
validation accuracy?

In [ ]:
try:
    torch.manual_seed(0)
    RUNS["augmented"] = fit(make_cnn(2, True), lr=0.05, epochs=EPOCHS, augment=augment, decay=True, label="augmented")
    plot_runs(RUNS, ["no augmentation", "augmented"], [NAVY, GOLD],
              f"6 conv + BN, {EPOCHS} epochs: augmentation narrows the gap", train_acc=True)
    for lab in ("no augmentation", "augmented"):
        h = RUNS[lab]
        print(f"{lab:<16} train {h['train_acc'][-1]:5.1%}   val {h['val_acc'][-1]:5.1%}   "
              f"gap {h['train_acc'][-1] - h['val_acc'][-1]:+5.1%}   best val {max(h['val_acc']):5.1%}")
except (NotImplementedError, KeyError):
    print("Complete Tasks 1–3 (and Step 1 above) first.")

### Checkpoint 3

**Q3a.** Augmentation made **training** accuracy *lower* and validation accuracy higher.
Why is lower training accuracy expected here, and not a sign that something broke? (What does
the network see in epoch 10 with augmentation that it never saw without it?)

*Your answer:*

**Q3b.** Augmentations must keep the label true. For CIFAR-10, would you add an
**up-down flip**? A **90° rotation**? Now think about the 1958 air photos in your project: which
of these become *safe* there, which new one would you add for photos from different eras, and is
there one that is still unsafe? (Hint: is every photo in the project data taken looking straight down?)

*Your answer:*

**Q3c.** *A worked example.* Suppose a team had chosen Project 7 (forest composition, which no group
took this year). They want a coniferous-vs-deciduous classifier for the 1958 photos, but only
the 2020 colour images are easy to label. They convert every labelled 2020
patch to greyscale (the label doesn't change). Then, every epoch, they jitter its brightness,
contrast and gamma, blur it slightly and add film grain, so that the training patches cover more
of the variety in the 1958 scans.

(i) What can this augmentation **not** give them that real labelled 1958 patches would? Which
patches should they use for `val` and `test`?

(ii) Your own project: could it benefit from data augmentation? Which transformations would you
use, and why do they keep your labels true?

*Your answer:*

---
## Part 4 — The test set, once  `⏱ ~5 min`

Every choice so far (learning rate, depth, batch norm, augmentation) was made by looking at
`val`. That makes `val` accuracy slightly **optimistic**: we kept whatever happened to score
well on those 2,000 images. The test set has played no part in any decision. We look at it
**once**, with the model we chose.

In [ ]:
try:
    final = RUNS["augmented"]["model"]
    pred = predict(final, Xte).cpu().numpy(); truth = yte.cpu().numpy()
    print(f"chosen model: 6 conv + BN + augmentation, lr 0.05 decaying to 0, {EPOCHS} epochs")
    print(f"val accuracy  {accuracy(final, Xva, yva):5.1%}")
    print(f"TEST accuracy {np.mean(pred == truth):5.1%}   (2,000 images, 200 per class)")
except KeyError:
    print("Complete Part 3 first.")

In [ ]:
if "augmented" in RUNS:
    CM = np.zeros((10, 10), int)
    np.add.at(CM, (truth, pred), 1)
    fig, ax = plt.subplots(1, 2, figsize=(16, 6.3), gridspec_kw={"width_ratios": [1.25, 1]})
    ax[0].imshow(CM, cmap="Blues")
    for i in range(10):
        for j in range(10):
            ax[0].text(j, i, CM[i, j], ha="center", va="center", fontsize=9,
                       color="white" if CM[i, j] > 100 else "black")
    ax[0].set_xticks(range(10), CLASSES, rotation=45, ha="right"); ax[0].set_yticks(range(10), CLASSES)
    ax[0].set_xlabel("predicted"); ax[0].set_ylabel("true"); ax[0].set_title("Confusion matrix on test (200 per row)")
    per_class = CM.diagonal() / CM.sum(1)
    order = np.argsort(per_class)
    ax[1].barh(range(10), per_class[order], color=[RED if p < 0.6 else TEAL for p in per_class[order]])
    ax[1].set_yticks(range(10), [CLASSES[i] for i in order]); ax[1].set_xlim(0, 1)
    ax[1].set_xlabel("test accuracy"); ax[1].set_title("Per class")
    for r, p in enumerate(per_class[order]):
        ax[1].text(p + 0.01, r, f"{p:.0%}", va="center")
    plt.tight_layout(); plt.show()
    off = CM - np.diag(CM.diagonal())
    top = np.dstack(np.unravel_index(np.argsort(-off.ravel())[:4], off.shape))[0]
    print("most common mistakes:", ";  ".join(f"{CLASSES[i]} -> {CLASSES[j]} ({off[i, j]})" for i, j in top))

In [ ]:
# The 12 most *confident* mistakes on test, then the 16 images from Part 0.
if "augmented" in RUNS:
    with torch.no_grad():
        final.eval()
        P = torch.cat([final(Xte[i:i + 500]).softmax(1) for i in range(0, len(Xte), 500)]).cpu().numpy()
    wrong = np.where(pred != truth)[0]
    worst = wrong[np.argsort(-P[wrong, pred[wrong]])][:12]
    fig, axes = plt.subplots(2, 6, figsize=(16, 7), gridspec_kw={"hspace": 0.35})
    for k, ax in zip(worst, axes.ravel()):
        ax.imshow(D["x_test"][k], interpolation="nearest"); ax.axis("off")
        ax.set_title(f"said {CLASSES[pred[k]]} ({P[k, pred[k]]:.0%})\nreally {CLASSES[truth[k]]}", fontsize=10, color=RED)
    plt.suptitle("Confidently wrong: the 12 test mistakes with the highest predicted probability", y=0.98)
    plt.show()

    hp = predict(final, Xva[torch.as_tensor(HOOK, device=DEVICE)]).cpu().numpy(); ht = D["y_val"][HOOK]
    fig, axes = plt.subplots(2, 8, figsize=(16, 5.2), gridspec_kw={"hspace": 0.3})
    for n, (k, ax) in enumerate(zip(HOOK, axes.ravel())):
        ax.imshow(D["x_val"][k], interpolation="nearest"); ax.axis("off")
        ax.set_title(f"#{n + 1} {CLASSES[hp[n]]}", fontsize=11, color=TEAL if hp[n] == ht[n] else RED)
    plt.suptitle(f"Part 0's 16 images: the network got {np.sum(hp == ht)} of 16 (red = wrong). How did you do?", y=0.98)
    plt.show()

### Checkpoint 4

**Q4a.** Which pairs of classes does the network confuse most? Do they make sense
for 32 × 32 photos? Look at the confident mistakes: name one that you would also have got wrong,
and one that no person would.

*Your answer:*

**Q4b.** Suppose you now tried five more learning rates and kept whichever scored best
on **test**. Why would the test number you report then be too optimistic? What should you do
instead?

*Your answer:*

---
## ★ Bonus — B1 · Forgetting `model.eval()`

During training, `BatchNorm2d` normalises with the mean and spread of the **current
mini-batch**, and quietly keeps a running average of them. After `model.eval()` it uses
those running averages instead, so each image's prediction no longer depends on which other
images happen to share its batch.

What if you forget `model.eval()` and predict in small batches, say 8 photos at a time?
We work on a **copy**: in training mode, every forward pass also *updates* the running
averages, so the mistake damages the model itself.

In [ ]:
if "augmented" in RUNS:
    m = copy.deepcopy(RUNS["augmented"]["model"])
    first_bn = next(l for l in m if isinstance(l, nn.BatchNorm2d))
    before = first_bn.running_mean[:4].cpu().numpy().copy()

    print(f"model.eval(),  batches of 500: val accuracy {accuracy(m, Xva, yva):5.1%}")
    m.train()                                            # the mistake
    with torch.no_grad():
        for bs in (500, 8, 2):
            right = sum((m(Xva[i:i + bs]).argmax(1) == yva[i:i + bs]).sum().item() for i in range(0, len(Xva), bs))
            print(f"model.train(), batches of {bs:3}: val accuracy {right / len(Xva):5.1%}")
    print("first BN layer's running mean, channels 0–3: before", before.round(3),
          " after", first_bn.running_mean[:4].cpu().numpy().round(3))
    print(f"after the mistake, back in eval mode: val accuracy {accuracy(m, Xva, yva):5.1%}")

**B1.** Why does accuracy fall as the batch gets smaller in training mode? And why did
the model get worse even after we switched back to `model.eval()`? Where in `fit` would this bug
bite if `accuracy` didn't call `model.eval()`?

*Your answer:*

---
## Reflection — End of Lab

**R1.** You start a new image classifier tomorrow. List, in order, the first three things
you would check or tune, using today's results as your evidence.

*Your answer:*

**R2.** Today's best network reached about three quarters accuracy on test. Name two
changes that you would expect to raise that the most, and say which of today's experiments
supports each one.

*Your answer:*

**R3.** Your project imagery has no CIFAR-style labels. If you trained a CNN for your
project (forest type, buildings, fields, shoreline, urban), what would one training example
be, how many could your team label by hand in an afternoon, and how would that number change
which of today's three knobs matters most? (AL8 on Friday Nov 13 and L14 on Monday take this
further: start from a network that has already seen a million photos.)

*Your answer:*

---
## ✏️ Try at home

Not graded.

**A. All 50,000 images.** `torchvision.datasets.CIFAR10(root=".", train=True, download=True)`
downloads the full set (~170 MB) on Colab. Train `make_cnn(2, True)` with augmentation for 30
epochs on a GPU. How much does five times the data add, compared to augmentation alone?

**B. A skip connection.** Write a residual block (L13 §3): `out = ReLU(x + BN(conv(ReLU(BN(conv(x))))))`.
Build a network of 4, then 8, of them per stage. Does depth keep paying, now that the
gradient has a shortcut?

**C. Your own photo.** Take a photo of something in one of the 10 classes, crop it square,
shrink it to 32 × 32 (`cv2.resize(..., interpolation=cv2.INTER_AREA)`), normalise it with
`MEAN` and `STD`, and ask `final`. Then try a photo of something that is in **none** of the 10
classes. What does the network say, and how confident is it?